# Lesson 5: Logistic Regression (Classification)

So far we predicted **numbers** (regression). Now we predict **categories** — e.g. will a student **pass (1)** or **fail (0)** based on hours studied?

Key new ideas in this lesson:
1. The **sigmoid** function — squashes any number into a probability between 0 and 1.
2. A **new loss function** (log loss / cross-entropy) — MSE doesn't work well here.
3. A **decision boundary** — turning a probability into a yes/no answer.

> Upload to [Google Colab](https://colab.research.google.com) and run top to bottom.

## Step 1: Why not just use linear regression?

Linear regression outputs *any* number: -3, 0.5, 47, ... But for pass/fail we need a **probability** between 0 and 1. A straight line shoots past 1 and below 0, which is meaningless as a probability.

So we wrap the line in a function that squashes it into [0, 1]: the **sigmoid**.

$$ \sigma(z) = \frac{1}{1 + e^{-z}} \qquad \text{where } z = w x + b $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 200)
plt.plot(z, sigmoid(z))
plt.axhline(0.5, color='red', linestyle='--', label='0.5 threshold')
plt.axvline(0, color='gray', linestyle=':')
plt.xlabel('z = w*x + b')
plt.ylabel('probability sigmoid(z)')
plt.title('The Sigmoid: squashes any number into (0, 1)')
plt.legend()
plt.grid(True)
plt.show()

**Read the S-curve:**
- Big positive `z` -> probability near **1** (confident PASS)
- Big negative `z` -> probability near **0** (confident FAIL)
- `z = 0` -> probability **0.5** (totally unsure)

## Step 2: A new loss function (why MSE is out)

For classification we use **Log Loss** (a.k.a. Binary Cross-Entropy):

$$ L = -\frac{1}{n}\sum \big[ y\,\log(p) + (1-y)\,\log(1-p) \big] $$

Intuition: it gives a **tiny** penalty when the predicted probability `p` is close to the true label, and a **huge** penalty when the model is confidently wrong (e.g. predicts 0.99 when the truth is 0). This shape pairs perfectly with the sigmoid for gradient descent.

In [ ]:
def log_loss(y_true, p):
    p = np.clip(p, 1e-9, 1 - 1e-9)   # avoid log(0)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

# Confident & correct vs confident & wrong
print('Truth=1, predicted 0.99 (correct, confident) loss:', round(log_loss(np.array([1]), np.array([0.99])), 4))
print('Truth=1, predicted 0.01 (wrong,   confident) loss:', round(log_loss(np.array([1]), np.array([0.01])), 4))

## Step 3: Train a logistic regression on pass/fail data

Same `.fit()` / `.predict()` pattern. Notice gradient descent is doing the work under the hood — there is **no closed-form formula** here.

In [ ]:
from sklearn.linear_model import LogisticRegression

# hours studied
X = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]).reshape(-1, 1)
# pass (1) or fail (0)
y = np.array([0, 0, 0, 0, 1, 0, 1, 1, 1, 1])

clf = LogisticRegression()
clf.fit(X, y)

print(f'w (coef)      = {clf.coef_[0][0]:.3f}')
print(f'b (intercept) = {clf.intercept_[0]:.3f}')

# Probability of passing for a few study times
for h in [2, 5, 8]:
    prob = clf.predict_proba([[h]])[0][1]
    label = clf.predict([[h]])[0]
    print(f'{h} hours -> P(pass)={prob:.2f} -> predicts {"PASS" if label==1 else "FAIL"}')

## Step 4: Visualize the fitted S-curve & decision boundary

In [ ]:
x_line = np.linspace(0, 11, 200).reshape(-1, 1)
probs = clf.predict_proba(x_line)[:, 1]

plt.scatter(X, y, color='blue', label='Actual (0=fail, 1=pass)', zorder=3)
plt.plot(x_line, probs, color='green', label='P(pass)')
plt.axhline(0.5, color='red', linestyle='--', label='0.5 decision threshold')

# decision boundary: where probability crosses 0.5  ->  z = 0  ->  x = -b/w
boundary = -clf.intercept_[0] / clf.coef_[0][0]
plt.axvline(boundary, color='purple', linestyle=':', label=f'boundary x={boundary:.2f}')

plt.xlabel('Hours studied')
plt.ylabel('Probability of passing')
plt.title('Logistic Regression: pass/fail')
plt.legend()
plt.grid(True)
plt.show()

print(f'Students studying more than {boundary:.2f} hours are predicted to PASS.')

## Your turn

1. Change the threshold from 0.5 to 0.7 (only predict PASS if P(pass) > 0.7). Which students flip? (This previews Lesson 6 — precision vs recall.)
2. Add noisy data (a student who studied 9 hours but failed). How does the curve react?
3. In your own words: why can't we use MSE here, and why is the sigmoid necessary?

## Bonus: Changing the threshold in code

`clf.predict()` always uses a hidden 0.5 cutoff. To use your own threshold, get probabilities with `predict_proba()` and compare manually.

In [ ]:
X_test = np.array([2, 5, 8]).reshape(-1, 1)

# P(pass) is column 1 of predict_proba
probs = clf.predict_proba(X_test)[:, 1]

for threshold in [0.4, 0.5, 0.7]:
    preds = (probs >= threshold).astype(int)
    print(f'--- threshold = {threshold} ---')
    for h, p, pred in zip(X_test.ravel(), probs, preds):
        print(f'  {h} hours -> P(pass)={p:.2f} -> {"PASS" if pred==1 else "FAIL"}')